# Step 2: Data Preprocessing & Exploratory Data Analysis (EDA)

## ⚖️ Project: Comparative Study of Word Embedding Techniques for Legal Document Classification

### Objective
The goal of this notebook is to implement a robust NLP preprocessing pipeline specialized for legal text. We will handle noise, normalize the text, and prepare the dataset for various word embedding techniques.

---

## 1. Environment Setup & Imports

We start by importing necessary libraries for data manipulation, NLP, and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from collections import Counter

# Add the src directory to path for modular imports
sys.path.append('..')
from src.preprocessing.preprocess import TextPreprocessor
from src.utils.helpers import ensure_dir

# Set aesthetic parameters
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

## 2. Dataset Loading & Inspection

Understanding the data is the first step in any NLP project. We need to know the distribution of classes, the length of documents, and the overall structure of the text.

In [ ]:
DATA_PATH = '../data/case_files_total.csv'
df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")
display(df.head())
print("\nColumn Info:")
print(df.info())

## 3. Data Cleaning

NLP models are sensitive to noise. We must handle missing values and duplicates to ensure the model learns from high-quality, unique examples.

In [ ]:
# Drop nulls in the target or text columns
initial_len = len(df)
df = df.dropna(subset=['judgement', 'case_category'])
print(f"Dropped {initial_len - len(df)} rows with missing values.")

# Remove duplicates
df = df.drop_duplicates(subset=['judgement'])
print(f"Dataset size after duplicate removal: {len(df)}")

## 4. Text Preprocessing Pipeline

We use our modular `TextPreprocessor` to clean the legal text. 

### Why Preprocessing Matters in Legal NLP?
Legal documents contain archaic language, heavy citations, and specific formatting (e.g., "Article 21", "Section 144"). Standard preprocessing must be adjusted to either preserve or carefully normalize these tokens.

In [ ]:
tp = TextPreprocessor()

# Sample cleaning
sample = df['judgement'].iloc[0][:300]
print("--- Original Text ---")
print(sample)

print("\n--- Cleaned & Normalized ---")
print(tp.full_preprocess(sample))

### Applying Pipeline to the entire Dataset
*Note: For large datasets, this might take a few minutes.*

In [ ]:
from tqdm import tqdm
tqdm.pandas()

print("Cleaning text data...")
df['processed_text'] = df['judgement'].progress_apply(tp.full_preprocess)

## 5. Tokenization & Legal Stopwords Analysis

**Stopword Removal**: In legal NLP, removing words like 'shall' or 'may' can be controversial because they change the legal weight of a sentence. However, for document-level classification, they often occur across all classes and add little discriminative value.

In [ ]:
words = " ".join(df['processed_text']).split()
word_freq = Counter(words)
print("Most common words:", word_freq.most_common(10))

## 6. Comparison: Stemming vs. Lemmatization

- **Stemming**: Chops off suffixes (e.g., "argued" -> "argu").
- **Lemmatization**: Uses a dictionary to find the root word (e.g., "argued" -> "argue").

Lemmatization is generally preferred for legal research as it maintains the semantic integrity of professional terms.

In [ ]:
test_words = ["appealed", "appealing", "appeals", "judge", "judgement", "judging"]
print(f"{'Word':<15} | {'Stemming':<15} | {'Lemmatization':<15}")
print("-"*50)
for w in test_words:
    print(f"{w:<15} | {tp.stem([w])[0]:<15} | {tp.lemmatizer.lemmatize(w):<15}")

## 7. Advanced NLP: POS Tagging & Dependency Parsing

POS tagging helps us identify the role of each word (Noun vs. Verb). Dependency parsing shows the relationship between legal actors and their actions.

In [ ]:
sample_phrase = "The Supreme Court dismissed the appeal filed by the defendant."
pos_tags = tp.get_pos_tags(sample_phrase)
print("POS Tags:", pos_tags)

print("\nDependency Parse:")
for item in tp.get_dependency_parsing(sample_phrase):
    print(f"{item[0]} -> {item[1]} -> {item[2]}")

## 8. Visualizations & Insights

We now visualize the processed data to understand class balance and word distribution.

In [ ]:
ensure_dir('../outputs/figures/')

# 1. Class Distribution
plt.figure(figsize=(10, 6))
sns.countplot(y='case_category', data=df, palette='viridis')
plt.title('Distribution of Legal Case Categories')
plt.savefig('../outputs/figures/class_distribution.png')
plt.show()

# 2. Text Length Distribution
df['text_len'] = df['processed_text'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10, 6))
sns.histplot(df['text_len'], bins=50, kde=True, color='teal')
plt.title('Distribution of Processed Text Length (Word Count)')
plt.savefig('../outputs/figures/text_length_dist.png')
plt.show()

## 9. Saving Cleaned Dataset

Finally, we save the cleaned dataset for use in the embedding comparison phase.

In [ ]:
OUTPUT_PATH = '../data/processed_legal_dataset.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSUCCESS: Cleaned dataset saved to {OUTPUT_PATH}")